# 04 — Evaluation

## 1. Why Evaluation?

We need labeled examples to measure how well our NLI predictions match the expected results.

## 2. Create Evaluation Dataset

In [15]:
data = [
    {
        "evidence": "The James Webb Space Telescope was launched on December 25, 2021.",
        "claim": "The telescope was launched in 2021.",
        "label": "ENTAILMENT"
    },
    {
        "evidence": "The James Webb Space Telescope was launched on December 25, 2021.",
        "claim": "The telescope was launched in 2020.",
        "label": "CONTRADICTION"
    },
    {
        "evidence": "The James Webb Space Telescope was launched on December 25, 2021.",
        "claim": "The telescope is used to study distant galaxies.",
        "label": "NEUTRAL"
    },
    {
        "evidence": "The telescope operates near the second Lagrange point.",
        "claim": "The telescope operates near the second Lagrange point.",
        "label": "ENTAILMENT"
    },
    {
        "evidence": "The telescope operates near the second Lagrange point.",
        "claim": "The telescope operates near Mars.",
        "label": "CONTRADICTION"
    },
    {
        "evidence": "The telescope operates near the second Lagrange point.",
        "claim": "The telescope was designed by NASA.",
        "label": "NEUTRAL"
    },
    {
        "evidence": "The telescope was launched aboard an Ariane 5 rocket.",
        "claim": "An Ariane 5 rocket launched the telescope.",
        "label": "ENTAILMENT"
    },
    {
        "evidence": "The telescope was launched aboard an Ariane 5 rocket.",
        "claim": "A Falcon 9 rocket launched the telescope.",
        "label": "CONTRADICTION"
    },
    {
        "evidence": "The telescope was launched aboard an Ariane 5 rocket.",
        "claim": "The telescope has infrared instruments.",
        "label": "NEUTRAL"
    },
    {
        "evidence": "The telescope operates approximately 1.5 million kilometers from Earth.",
        "claim": "The telescope operates approximately 1.5 million kilometers from Earth.",
        "label": "ENTAILMENT"
    }
]

## 3. View the Dataset

In [16]:
import pandas as pd
df = pd.DataFrame(data)
df

,evidence,claim,label
0,The James Webb Space Telescope was launched on...,The telescope was launched in 2021.,ENTAILMENT
1,The James Webb Space Telescope was launched on...,The telescope was launched in 2020.,CONTRADICTION
2,The James Webb Space Telescope was launched on...,The telescope is used to study distant galaxies.,NEUTRAL
3,The telescope operates near the second Lagrang...,The telescope operates near the second Lagrang...,ENTAILMENT
4,The telescope operates near the second Lagrang...,The telescope operates near Mars.,CONTRADICTION
5,The telescope operates near the second Lagrang...,The telescope was designed by NASA.,NEUTRAL
6,The telescope was launched aboard an Ariane 5 ...,An Ariane 5 rocket launched the telescope.,ENTAILMENT
7,The telescope was launched aboard an Ariane 5 ...,A Falcon 9 rocket launched the telescope.,CONTRADICTION
8,The telescope was launched aboard an Ariane 5 ...,The telescope has infrared instruments.,NEUTRAL
9,The telescope operates approximately 1.5 milli...,The telescope operates approximately 1.5 milli...,ENTAILMENT


## 4. Check Label Distribution

In [17]:
df["label"].value_counts()

label
ENTAILMENT       4
CONTRADICTION    3
NEUTRAL          3
Name: count, dtype: int64

## 5. Generate Predictions

In [18]:
from transformers import pipeline
nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768"
)
predictions = []
for _, row in df.iterrows():
    result = nli(
        f"{row['evidence']} </s></s> {row['claim']}"
    )
    predictions.append(result[0]["label"].upper())
df["prediction"] = predictions
df

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,evidence,claim,label,prediction
0,The James Webb Space Telescope was launched on...,The telescope was launched in 2021.,ENTAILMENT,ENTAILMENT
1,The James Webb Space Telescope was launched on...,The telescope was launched in 2020.,CONTRADICTION,CONTRADICTION
2,The James Webb Space Telescope was launched on...,The telescope is used to study distant galaxies.,NEUTRAL,NEUTRAL
3,The telescope operates near the second Lagrang...,The telescope operates near the second Lagrang...,ENTAILMENT,ENTAILMENT
4,The telescope operates near the second Lagrang...,The telescope operates near Mars.,CONTRADICTION,CONTRADICTION
5,The telescope operates near the second Lagrang...,The telescope was designed by NASA.,NEUTRAL,NEUTRAL
6,The telescope was launched aboard an Ariane 5 ...,An Ariane 5 rocket launched the telescope.,ENTAILMENT,ENTAILMENT
7,The telescope was launched aboard an Ariane 5 ...,A Falcon 9 rocket launched the telescope.,CONTRADICTION,CONTRADICTION
8,The telescope was launched aboard an Ariane 5 ...,The telescope has infrared instruments.,NEUTRAL,NEUTRAL
9,The telescope operates approximately 1.5 milli...,The telescope operates approximately 1.5 milli...,ENTAILMENT,ENTAILMENT


## 6. Compare Results

In [19]:
df["correct"] = df["label"] == df["prediction"]
df[["claim", "label", "prediction", "correct"]]

,claim,label,prediction,correct
0,The telescope was launched in 2021.,ENTAILMENT,ENTAILMENT,True
1,The telescope was launched in 2020.,CONTRADICTION,CONTRADICTION,True
2,The telescope is used to study distant galaxies.,NEUTRAL,NEUTRAL,True
3,The telescope operates near the second Lagrang...,ENTAILMENT,ENTAILMENT,True
4,The telescope operates near Mars.,CONTRADICTION,CONTRADICTION,True
5,The telescope was designed by NASA.,NEUTRAL,NEUTRAL,True
6,An Ariane 5 rocket launched the telescope.,ENTAILMENT,ENTAILMENT,True
7,A Falcon 9 rocket launched the telescope.,CONTRADICTION,CONTRADICTION,True
8,The telescope has infrared instruments.,NEUTRAL,NEUTRAL,True
9,The telescope operates approximately 1.5 milli...,ENTAILMENT,ENTAILMENT,True


## 7. Calculate Accuracy

In [20]:
accuracy = df["correct"].mean()
print("Accuracy:", accuracy)

Accuracy: 1.0


## 8. Precision, Recall and F1

In [21]:
from sklearn.metrics import classification_report
print(
    classification_report(
        df["label"],
        df["prediction"]
    )
)

               precision    recall  f1-score   support

CONTRADICTION       1.00      1.00      1.00         3
   ENTAILMENT       1.00      1.00      1.00         4
      NEUTRAL       1.00      1.00      1.00         3

     accuracy                           1.00        10
    macro avg       1.00      1.00      1.00        10
 weighted avg       1.00      1.00      1.00        10



## 9. Confusion Matrix

In [22]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    df["label"],
    df["prediction"],
    labels=["ENTAILMENT", "CONTRADICTION", "NEUTRAL"]
)
print(cm)

[[4 0 0]
 [0 3 0]
 [0 0 3]]


## 10. Error Analysis

In [23]:
errors = df[df["correct"] == False]
errors[["evidence", "claim", "label", "prediction"]]

,evidence,claim,label,prediction


## 11. Observations
The NLI model does not classify every example correctly. 
The errors show where the model may struggle with the relationship between evidence and claims.